# E2

La idea es completar variables faltantes usando modelos supervisados:

- Mision 1: predecir `PM2.5`.
- Mision 2: predecir `O3`.
- Mision 3: predecir `Environmental_risk`.

Para evitar data leakage, el conjunto de test se reserva solo para evaluacion final. 

La eleccion entre modelos se hace con un conjunto de validacion creado desde los datos de entrenamiento.

In [1]:
import numpy as np
import pandas as pd


from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression


from sklearn.metrics import (
    balanced_accuracy_score,
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
)

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


### carga y limpieza inicial

In [2]:
df_original = pd.read_csv("data/E2_datos.csv")
display(df_original.head())
display(df_original.info())

,Year,Month,Day,O3,PM2.5,Environmental_risk
0,2008,1,1,29.63,NaN,NaN
1,2008,1,2,21.46,NaN,NaN
2,2008,1,3,24.25,NaN,NaN
3,2008,1,4,29.04,NaN,NaN
4,2008,1,5,30.17,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 2984 entries, 0 to 2983
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Year                2984 non-null   int64  
 1   Month               2984 non-null   int64  
 2   Day                 2984 non-null   int64  
 3   O3                  2878 non-null   float64
 4   PM2.5               2725 non-null   float64
 5   Environmental_risk  2656 non-null   str    
dtypes: float64(2), int64(3), str(1)
memory usage: 140.0 KB


None

Hay valores `"nulo"` en la columna `Environmental_risk`. Los reemplazamos por `NaN` para tratarlos como faltantes reales.

In [3]:
df = df_original.replace("nulo", np.nan).copy()

missing_initial = df.isna().sum()
display(missing_initial.rename("faltantes_iniciales"))
display(df["Environmental_risk"].value_counts(dropna=False).rename("conteo_riesgo"))

Year                    0
Month                   0
Day                     0
O3                    106
PM2.5                 259
Environmental_risk    462
Name: faltantes_iniciales, dtype: int64

Environmental_risk
medio      1605
Bajo        750
NaN         462
Alto        106
extremo      61
Name: conteo_riesgo, dtype: int64

Hay filas donde faltan simultaneamente `O3` y `PM2.5`. Con las reglas del enunciado no se pueden completar, porque para predecir `PM2.5` se necesita `O3`, y para predecir `O3` se necesita `PM2.5`.

Por eso se eliminan esas filas antes de las misiones. Es una perdida chica respecto del total y permite cumplir que despues de la Mision 2 solo queden faltantes en `Environmental_risk`.

In [4]:
mask_both_pollutants_missing = df["O3"].isna() & df["PM2.5"].isna()

print(f"Filas originales: {len(df)}")
print(f"Filas con O3 y PM2.5 faltantes a la vez: {int(mask_both_pollutants_missing.sum())}")

df = df.loc[~mask_both_pollutants_missing].copy()

print(f"Filas luego de eliminar bloqueo circular: {len(df)}")
display(df.isna().sum().rename("faltantes_despues_limpieza_inicial"))

Filas originales: 2984
Filas con O3 y PM2.5 faltantes a la vez: 37
Filas luego de eliminar bloqueo circular: 2947


Year                    0
Month                   0
Day                     0
O3                     69
PM2.5                 222
Environmental_risk    425
Name: faltantes_despues_limpieza_inicial, dtype: int64

## funciones auxiliares para evitar leakage

In [ ]:
date_features = ["Year", "Month", "Day"]

def split_train_validation_test(X, y, stratify=None):
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=stratify,
    )

    if stratify is not None:
        stratify_train_val = y_train_val
    else:
        stratify_train_val = None

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=0.25,
        random_state=42,
        stratify=stratify_train_val,
    )

    return X_train, X_val, X_train_val, X_test, y_train, y_val, y_train_val, y_test

def regression_preprocessor(numeric_feature):
    return ColumnTransformer(
        transformers=[
            ("fecha", OneHotEncoder(handle_unknown="ignore", sparse_output=False), date_features),
            ("numerica", StandardScaler(), [numeric_feature]),
        ]
    )

def make_regression_pipeline(model, numeric_feature):
    regressor = Pipeline(
        steps=[
            ("preprocess", regression_preprocessor(numeric_feature)),
            ("model", model),
        ]
    )

    return TransformedTargetRegressor( # para aplicar una transformacion al target "y" durante el training y luego devolver las predicciones a la escala original
        regressor=regressor,
        transformer=StandardScaler(),
    )

    # en el fondo, es una alternativa a
    
    # scaler_y = StandardScaler()
    # y_train_scaled = scaler_y.fit_transform(y_train)

    # modelo.fit(X_train, y_train_scaled)

    # pred_scaled = modelo.predict(X_test)
    # pred = scaler_y.inverse_transform(pred_scaled)

def regression_metrics(y_true, predictions):
    return {
        "MSE": mean_squared_error(y_true, predictions),
        "MAE": mean_absolute_error(y_true, predictions),
        "MAPE": mean_absolute_percentage_error(y_true, predictions),
    }

## Mision 1: prediccion de PM2.5

Usamos solo filas completas en `Year`, `Month`, `Day`, `O3` y `PM2.5`. 

Las variables de fecha se tratan como categoricas y `O3` se normaliza dentro del pipeline. 

El target `PM2.5` tambien se normaliza usando `TransformedTargetRegressor`, ajustado solo con los datos de entrenamiento.

In [6]:
df_pm = df.dropna(subset=["Year", "Month", "Day", "O3", "PM2.5"]).copy()

X_pm = df_pm[["Year", "Month", "Day", "O3"]]
y_pm = df_pm["PM2.5"]

(   X_train_pm,
    X_val_pm,
    X_train_val_pm,
    X_test_pm,
    y_train_pm,
    y_val_pm,
    y_train_val_pm,
    y_test_pm,
) = split_train_validation_test(X_pm, y_pm)

print(f"filas completas para modelar PM2.5: {len(df_pm)}")
print(f"train: {len(X_train_pm)}, validacion: {len(X_val_pm)}, test: {len(X_test_pm)}")

filas completas para modelar PM2.5: 2656
train: 1593, validacion: 531, test: 532


In [7]:
regression_candidates = {
    "LinearRegression": LinearRegression(),
    "KNeighborsRegressor": KNeighborsRegressor(n_neighbors=5),
}

pm_validation_results = []

for name, base_model in regression_candidates.items():
    model = make_regression_pipeline(clone(base_model), numeric_feature="O3")
    model.fit(X_train_pm, y_train_pm)
    val_predictions = model.predict(X_val_pm)
    metrics_val = regression_metrics(y_val_pm, val_predictions)
    pm_validation_results.append({"modelo": name, **metrics_val})

pm_validation_results = (pd.DataFrame(pm_validation_results).sort_values("MSE", ascending=True).reset_index(drop=True))

display(pm_validation_results)

,modelo,MSE,MAE,MAPE
0,KNeighborsRegressor,178.255935,9.301107,0.396397
1,LinearRegression,183.750117,9.484528,0.413923


In [8]:
best_pm_name = pm_validation_results.loc[0, "modelo"]
best_pm_base_model = regression_candidates[best_pm_name]

final_pm_model = make_regression_pipeline(clone(best_pm_base_model), numeric_feature="O3") # clone para hacer una copia del modelo base y no modificar el original, por si se quiere usar despues para O3
final_pm_model.fit(X_train_val_pm, y_train_val_pm)

pm_test_predictions = final_pm_model.predict(X_test_pm)
pm_test_metrics = regression_metrics(y_test_pm, pm_test_predictions)

print(f"modelo elegido para PM2.5 usando validacion: {best_pm_name}")
display(pd.DataFrame([pm_test_metrics], index=["test"]))

modelo elegido para PM2.5 usando validacion: KNeighborsRegressor


,MSE,MAE,MAPE
test,178.971066,9.240688,0.394022


Ahora se usa la familia de modelo elegida para completar los faltantes de `PM2.5`. Para imputar, se reajusta ese modelo con todas las filas que si tienen `PM2.5` observado. Esto no cambia el rendimiento reportado en test; solo aprovecha mas datos para completar la base.

In [9]:
imputer_pm_model = make_regression_pipeline(clone(best_pm_base_model), numeric_feature="O3")
imputer_pm_model.fit(X_pm, y_pm)

mask_fill_pm = (
    df["PM2.5"].isna()
    & df["Year"].notna()
    & df["Month"].notna()
    & df["Day"].notna()
    & df["O3"].notna()
)

print(f"filas a completar en PM2.5: {int(mask_fill_pm.sum())}")

if mask_fill_pm.any():
    X_fill_pm = df.loc[mask_fill_pm, ["Year", "Month", "Day", "O3"]]
    df.loc[mask_fill_pm, "PM2.5"] = imputer_pm_model.predict(X_fill_pm)

print(f"faltantes en PM2.5 despues de imputacion: {int(df['PM2.5'].isna().sum())}")

filas a completar en PM2.5: 222
faltantes en PM2.5 despues de imputacion: 0


## Mision 2: prediccion de O3

Repetimos el procedimiento para `O3`, usando `PM2.5` como variable numerica predictora.

In [10]:
df_o3 = df.dropna(subset=["Year", "Month", "Day", "PM2.5", "O3"]).copy()

X_o3 = df_o3[["Year", "Month", "Day", "PM2.5"]]
y_o3 = df_o3["O3"]

(
    X_train_o3,
    X_val_o3,
    X_train_val_o3,
    X_test_o3,
    y_train_o3,
    y_val_o3,
    y_train_val_o3,
    y_test_o3,
) = split_train_validation_test(X_o3, y_o3)

print(f"filas completas para modelar O3: {len(df_o3)}")
print(f"train: {len(X_train_o3)}, Validacion: {len(X_val_o3)}, Test: {len(X_test_o3)}")

filas completas para modelar O3: 2878
train: 1726, Validacion: 576, Test: 576


In [11]:
o3_validation_results = []

for name, base_model in regression_candidates.items():
    model = make_regression_pipeline(clone(base_model), numeric_feature="PM2.5")
    model.fit(X_train_o3, y_train_o3)
    val_predictions = model.predict(X_val_o3)
    metrics_val = regression_metrics(y_val_o3, val_predictions)
    o3_validation_results.append({"modelo": name, **metrics_val})

o3_validation_results = (pd.DataFrame(o3_validation_results).sort_values("MSE", ascending=True).reset_index(drop=True))

display(o3_validation_results)

,modelo,MSE,MAE,MAPE
0,LinearRegression,22.054755,3.668748,0.386697
1,KNeighborsRegressor,23.963316,3.623170,0.381362


In [12]:
best_o3_name = o3_validation_results.loc[0, "modelo"]
best_o3_base_model = regression_candidates[best_o3_name]

final_o3_model = make_regression_pipeline(clone(best_o3_base_model), numeric_feature="PM2.5")
final_o3_model.fit(X_train_val_o3, y_train_val_o3)

o3_test_predictions = final_o3_model.predict(X_test_o3)
o3_test_metrics = regression_metrics(y_test_o3, o3_test_predictions)

print(f"modelo elegido para O3 usando validacion: {best_o3_name}")
display(pd.DataFrame([o3_test_metrics], index=["test"]))

modelo elegido para O3 usando validacion: LinearRegression


,MSE,MAE,MAPE
test,20.300952,3.470776,0.330238


In [13]:
imputer_o3_model = make_regression_pipeline(clone(best_o3_base_model), numeric_feature="PM2.5")
imputer_o3_model.fit(X_o3, y_o3)

mask_fill_o3 = (
    df["O3"].isna()
    & df["Year"].notna()
    & df["Month"].notna()
    & df["Day"].notna()
    & df["PM2.5"].notna()
)

print(f"filas a completar en O3: {int(mask_fill_o3.sum())}")

if mask_fill_o3.any():
    X_fill_o3 = df.loc[mask_fill_o3, ["Year", "Month", "Day", "PM2.5"]]
    df.loc[mask_fill_o3, "O3"] = imputer_o3_model.predict(X_fill_o3)

display(df.isna().sum().rename("faltantes_despues_misiones_1_y_2"))

filas a completar en O3: 69


Year                    0
Month                   0
Day                     0
O3                      0
PM2.5                   0
Environmental_risk    425
Name: faltantes_despues_misiones_1_y_2, dtype: int64

## Mision 3: prediccion de Environmental_risk

Con `O3` y `PM2.5` completos, se entrena un clasificador para completar `Environmental_risk`. De nuevo se elige el modelo usando validacion y se reporta balanced accuracy en test.


Usamos balanced accuracy porque calcula el recall por clase y luego promedia esos valores. Es más informativo que el accuracy global cuando hay clases desbalanceadas.

In [14]:
df_risk = df.dropna(subset=["Year", "Month", "Day", "O3", "PM2.5", "Environmental_risk"]).copy()

X_risk = df_risk[["Year", "Month", "Day", "O3", "PM2.5"]]
y_risk_labels = df_risk["Environmental_risk"]

risk_encoder = LabelEncoder()
y_risk = risk_encoder.fit_transform(y_risk_labels)

(
    X_train_risk,
    X_val_risk,
    X_train_val_risk,
    X_test_risk,
    y_train_risk,
    y_val_risk,
    y_train_val_risk,
    y_test_risk,
) = split_train_validation_test(X_risk, y_risk, stratify=y_risk)

print(f"clases de Environmental_risk: {list(risk_encoder.classes_)}")
print(f"filas completas para clasificacion: {len(df_risk)}")
print(f"train: {len(X_train_risk)}, Validacion: {len(X_val_risk)}, Test: {len(X_test_risk)}")

clases de Environmental_risk: ['Alto', 'Bajo', 'extremo', 'medio']
filas completas para clasificacion: 2522
train: 1512, Validacion: 505, Test: 505


In [15]:
def classification_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("fecha", OneHotEncoder(handle_unknown="ignore", sparse_output=False), date_features),
            ("contaminantes", StandardScaler(), ["O3", "PM2.5"]),
        ]
    )

def make_classification_pipeline(model):
    return Pipeline(
        steps=[
            ("preprocess", classification_preprocessor()),
            ("model", model),
        ]
    )

classifier_candidates = {
    "DecisionTreeClassifier": DecisionTreeClassifier(max_depth=6, random_state=42),
    "KNeighborsClassifier": KNeighborsClassifier(n_neighbors=5),
    "RandomForestClassifier": RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
}

In [16]:
risk_validation_results = []

for name, base_model in classifier_candidates.items():
    model = make_classification_pipeline(clone(base_model))
    model.fit(X_train_risk, y_train_risk)
    val_predictions = model.predict(X_val_risk)
    bal_acc = balanced_accuracy_score(y_val_risk, val_predictions)
    risk_validation_results.append({"modelo": name, "balanced_accuracy_validacion": bal_acc})

risk_validation_results = (pd.DataFrame(risk_validation_results).sort_values("balanced_accuracy_validacion", ascending=False).reset_index(drop=True))

display(risk_validation_results)

,modelo,balanced_accuracy_validacion
0,DecisionTreeClassifier,1.000000
1,LogisticRegression,0.888963
2,KNeighborsClassifier,0.792142
3,RandomForestClassifier,0.791667


In [17]:
best_risk_name = risk_validation_results.loc[0, "modelo"]
best_risk_base_model = classifier_candidates[best_risk_name]

final_risk_model = make_classification_pipeline(clone(best_risk_base_model))
final_risk_model.fit(X_train_val_risk, y_train_val_risk)

risk_test_predictions = final_risk_model.predict(X_test_risk)
risk_test_balanced_accuracy = balanced_accuracy_score(y_test_risk, risk_test_predictions)

print(f"modelo elegido para Environmental_risk usando validacion: {best_risk_name}")
print(f"balanced accuracy en test: {risk_test_balanced_accuracy:.4f}")

modelo elegido para Environmental_risk usando validacion: DecisionTreeClassifier
balanced accuracy en test: 0.9992


In [18]:
cm_risk = pd.DataFrame(
    pd.crosstab(
        risk_encoder.inverse_transform(y_test_risk),
        risk_encoder.inverse_transform(risk_test_predictions),
        rownames=["real"],
        colnames=["prediccion"],
    )
)

display(cm_risk)

prediccion,Alto,Bajo,extremo,medio
real,,,,
Alto,21,0,0,0
Bajo,0,150,0,0
extremo,0,0,12,0
medio,0,0,1,321


In [19]:
imputer_risk_model = make_classification_pipeline(clone(best_risk_base_model))
imputer_risk_model.fit(X_risk, y_risk)

mask_fill_risk = (
    df["Environmental_risk"].isna()
    & df["Year"].notna()
    & df["Month"].notna()
    & df["Day"].notna()
    & df["O3"].notna()
    & df["PM2.5"].notna()
)

print(f"Filas a completar en Environmental_risk: {int(mask_fill_risk.sum())}")

if mask_fill_risk.any():
    X_fill_risk = df.loc[mask_fill_risk, ["Year", "Month", "Day", "O3", "PM2.5"]]
    predicted_risk = imputer_risk_model.predict(X_fill_risk)
    df.loc[mask_fill_risk, "Environmental_risk"] = risk_encoder.inverse_transform(predicted_risk)

display(df.isna().sum().rename("faltantes_finales"))

Filas a completar en Environmental_risk: 425


Year                  0
Month                 0
Day                   0
O3                    0
PM2.5                 0
Environmental_risk    0
Name: faltantes_finales, dtype: int64

## Conclusion

Las decisiones de modelo se toman con validacion:

- para `PM2.5`, se comparan dos regresores usando MSE en validacion;
- para `O3`, se repite el mismo criterio;
- para `Environmental_risk`, se comparan clasificadores usando balanced accuracy en validacion.

Despues de las imputaciones, la base queda sin valores faltantes en las columnas conservadas.